In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)
base_temp = 9.5      # середньорічна температура Києва
amplitude = 14       # сезонна амплітуда Києва
city = "Київ"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)   # 48 рядків, 4 роки x 12 місяців
print(climate.head())
print("\nРозмір набору:", climate.shape)

In [ ]:
table = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
print("Сирі кількості:")
print(table)

print("\nНормована за рядками (частки всередині кожного сезону):")
table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize="index")
print(table_norm.round(3))

In [ ]:
chi2, pvalue, dof, expected = chi2_contingency(table)

print(f"chi2 = {chi2:.2f}")
print(f"p-value = {pvalue:.6f}")
print(f"dof = {dof}")

print("\nОчікувані частоти (expected):")
print(pd.DataFrame(expected, index=table.index, columns=table.columns).round(2))

print("\nФактичні частоти (observed):")
print(table)

# найбільша розбіжність
diff_table = table - pd.DataFrame(expected, index=table.index, columns=table.columns)
print("\nРізниця observed - expected:")
print(diff_table.round(2))

In [ ]:
expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)

print("Мінімальна очікувана частота:", expected_df.values.min().round(2))
print("\nКлітинки з expected < 5:")
mask = expected_df < 5
print(mask)

# Які саме клітинки не проходять поріг
low_cells = [(r, c, round(expected_df.loc[r, c], 2))
             for r in expected_df.index for c in expected_df.columns
             if expected_df.loc[r, c] < 5]
print("\nСписок проблемних клітинок (рядок, стовпець, expected):")
for cell in low_cells:
    print(cell)

In [ ]:
# Об'єднуємо категорії: холодне + звичайно -> "не тепліше"
climate["відхилення_об'єднане"] = climate["відхилення_від_норми"].replace(
    {"холодніше": "не тепліше", "звичайно": "не тепліше"}
)

table_merged = pd.crosstab(climate["сезон"], climate["відхилення_об'єднане"])
print("Об'єднана таблиця:")
print(table_merged)

chi2_m, pvalue_m, dof_m, expected_m = chi2_contingency(table_merged)
print(f"\nchi2 = {chi2_m:.2f}, p-value = {pvalue_m:.6f}, dof = {dof_m}")
print("\nНові expected:")
print(pd.DataFrame(expected_m, index=table_merged.index, columns=table_merged.columns).round(2))
print("\nМінімальна expected тепер:", expected_m.min().round(2))

In [ ]:
season_counts = climate["сезон"].value_counts()
print("Фактична частота сезонів:")
print(season_counts)

n = len(climate)
expected_seasons = [0.25 * n] * 4   # по 12 спостережень на сезон

result = chisquare(f_obs=season_counts.reindex(["зима", "весна", "літо", "осінь"]),
                   f_exp=expected_seasons)
print("\nСтатистика =", round(result.statistic, 4))
print("p-value =", round(result.pvalue, 4))

In [ ]:
cat_counts = climate["відхилення_від_норми"].value_counts().reindex(
    ["холодніше", "звичайно", "тепліше"])
print("Фактичні частоти:")
print(cat_counts)

n = len(climate)
my_expected = [0.33 * n, 0.34 * n, 0.33 * n]

result5 = chisquare(f_obs=cat_counts, f_exp=my_expected)
print("\nСтатистика =", round(result5.statistic, 4))
print("p-value =", round(result5.pvalue, 4))

## Контрольні питання

### 1. Механіка статистики χ²

χ² = Σ (O − E)² / E. Квадрат відхилення потрібен, щоб: (а) додатні й від'ємні відхилення не компенсували одне одного; (б) посилити штраф за великі відхилення. Ділення на E робить відхилення **відносним**: різниця в 5 одиниць важлива по-різному, якщо очікувалось 5 чи 100. Нормування на E приводить усі клітинки до спільної шкали, і сума таких стандартизованих квадратів за H0 прямує до розподілу χ².

### 2. Як обчислюється очікувана частота клітинки

E = (рядкова сума × стовпцева сума) / загальна сума. Під припущенням незалежності P(рядок І стовпець) = P(рядок)·P(стовпець). Множимо цю ймовірність на загальну кількість n: n · (рядок/n) · (стовпець/n) = рядок×стовпець/n. Саме тому формула коректна саме за незалежності — вона реалізує правило множення ймовірностей для незалежних подій.

### 3. «Немає підстав відхилити» ≠ «доведено незалежність»

Великий p-value лише означає, що **даних недостатньо**, щоб заперечити проти незалежності — це не доказ того,